# The Verb Requirement Test

### Predicting what a substrate must contain in order to latch or to count, from the verb list alone

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## What this notebook does

The claim under test is that learning does not add values to an object; it exposes arguments to verbs
that were already callable. `Sun()` becomes `Sun(a,b,c,d)` without the sun changing. If that is right,
then the *verb set* is substrate-independent and small, and the verbs carry requirements that are
substrate-independent too.

That claim is cheap until it is inverted. Forward-mapping is easy: point at a genetic toggle, point at
a flip-flop, say "same verb." Resemblance is free. The inverse is not free:

> Take a substrate whose mechanism was **not** used to build the verb list. From the verb requirements
> **alone**, predict what that substrate must contain in order to latch, and what it must contain in
> order to count. Then check.

The verb list was built from electrons — channels, gates, wells, rails, inverter rings. The test
substrate here is a **chemical reaction network**: no voltage, no rails, no channel, no supply. Just
species concentrations with production and decay.

The discipline this has to satisfy is the one that separates naming the right object from decorating
with a wrong one: **a right object arrives with a number that could have been otherwise and matches.**
A resemblance arrives with a number chosen to fit. So each prediction below has to be *falsifiable and
separately necessary* — knocking out one requirement must kill the capability while the others stay
intact.

---

## Step 1 — The predictions, written before touching the chemistry

From the electron-level verb map, **Latch** decomposes into these requirements:

| # | Verb requirement | Why the verb needs it | Prediction for chemistry |
|---|---|---|---|
| R1 | **Threshold** — a nonlinearity that snaps | Without it there is no distinction to hold (C0 is not manufactured) | Reaction must be **cooperative**: Hill exponent $n>1$. Mass-action ($n=1$) can never latch |
| R2 | **Gain** > 1 around the loop | Continuation must not decay (C1) | There is a **critical production rate** $a_c$. Below it, no memory, at any $n$ |
| R3 | **Positive loop** (mutual suppression) | Two attractors require the round trip to reinforce, not correct | Two **repressors** latch. Swap one for an activator — same gain, same threshold — and the latch dies |
| R4 | **Hold** — each state must resist decay | Law 2: no state held without a price | Species must **degrade**, so the held state is actively maintained, not inert |

And **Clock** needs everything Latch needs, plus one more:

| # | Verb requirement | Prediction for chemistry |
|---|---|---|
| R5 | **Negative loop with phase lag** | An odd-length inhibition ring. And a 2-species ring cannot do it — counting costs **one more dimension** than memory |

R1–R3 and R5 are all falsifiable. R1 and R2 have numbers attached. R5 has a parity. One of them fails as stated, and the failure turns out
to be the most useful result here.

Nothing below uses any knowledge of the genetic toggle switch or the repressilator. The model is written
from the requirement table, and then run.


---
## Step 2 — Library cell

Self-contained. Only numpy and scipy.


In [1]:
import numpy as np
from scipy.optimize import brentq
from scipy.integrate import solve_ivp

# ---------------------------------------------------------------------------
# The test substrate: a chemical reaction network.
# Species i is produced at rate a, inhibited by species j with cooperativity n,
# and degrades at rate 1 (time units chosen so decay rate = 1).
#
#     d[X_i]/dt = a / (1 + [X_j]^n)  -  [X_i]
#
# No voltages, no rails, no channel. Production, inhibition, decay.
# ---------------------------------------------------------------------------

def hill(x, a, n):
    """Inhibited production: high inhibitor -> low production. The Threshold verb, if n>1."""
    return a / (1.0 + x**n)


def toggle_rhs(t, y, a, n):
    """Two mutual repressors. Positive loop (two inversions). Candidate LATCH."""
    u, v = y
    return [hill(v, a, n) - u, hill(u, a, n) - v]


def mixed_rhs(t, y, a, n):
    """One repressor + one activator. Negative loop. Same gain, same threshold, loop sign flipped."""
    u, v = y
    return [hill(v, a, n) - u, a * u**n / (1.0 + u**n) - v]


def ring_rhs(t, y, a, n):
    """Ring of N repressors, each inhibiting the next. Loop sign = (-1)^N."""
    N = len(y)
    return [hill(y[(i - 1) % N], a, n) - y[i] for i in range(N)]


# --- fixed-point counting for the symmetric toggle -------------------------
# Steady state: u = f(f(u)) with f = hill. Three roots => bistable => latch.

def toggle_fixed_points(a, n, N=200000):
    g = lambda u: hill(hill(u, a, n), a, n) - u
    xs = np.linspace(1e-9, a, N)
    gs = g(xs)
    idx = np.nonzero(gs[:-1] * gs[1:] < 0)[0]
    return sorted(brentq(g, xs[i], xs[i + 1]) for i in idx)


def symmetric_fixed_point(a, n):
    """The single symmetric state u* = v*, solving u = f(u)."""
    return brentq(lambda u: hill(u, a, n) - u, 1e-14, a)


def loop_gain(a, n):
    """Round-trip gain: |f'(u*)|^2 for the two-stage loop. |f'| is the per-stage gain."""
    u = symmetric_fixed_point(a, n)
    per_stage = a * n * u**(n - 1) / (1.0 + u**n)**2
    return per_stage, per_stage**2


# --- closed form for the onset of bistability -----------------------------
# Derived in Step 4. Requires n > 1; diverges as n -> 1+.

def a_critical(n):
    return n * (n - 1.0)**(-(n + 1.0) / n)


# --- dynamics helpers -----------------------------------------------------

def settle(rhs, y0, a, n, T=4000.0):
    """Integrate to steady state, return the endpoint."""
    s = solve_ivp(rhs, [0, T], y0, args=(a, n), rtol=1e-11, atol=1e-13)
    return np.round(s.y[:, -1], 3)


def attractor_count(rhs, N, a, n, trials=12, seed=0, T=3000.0):
    """How many distinct attractors does this network settle into from random starts?"""
    rng = np.random.default_rng(seed)
    found = set()
    for _ in range(trials):
        y0 = list(rng.uniform(0.01, a, N))
        found.add(tuple(settle(rhs, y0, a, n, T)))
    return found


def oscillation_amplitude(N, a, n, T=4000.0):
    """Peak-to-peak swing in the last 15% of the run. Nonzero => sustained cycle."""
    y0 = list(np.linspace(0.5, 3.0, N))
    s = solve_ivp(ring_rhs, [0, T], y0, args=(a, n),
                  rtol=1e-12, atol=1e-14, dense_output=True)
    tail = s.sol(np.linspace(0.85 * T, T, 6000))
    return float((tail.max(axis=1) - tail.min(axis=1)).max())

print("library ready")


library ready


---
## Step 3 — R1: is the Threshold verb actually necessary?

Prediction: **no**, mass action cannot latch. Not "latches poorly" — cannot, at any production rate.
This is the strong form, so it is the easiest to falsify: one bistable case at $n=1$ and R1 is dead.

Push the gain to $10^6$ and sweep $n$ across 1.


In [2]:
print("R1  Threshold necessity.  Gain pushed to a = 1e6 throughout.\n")
print(f"{'n':>8}  {'# steady states':>16}   verdict")
print("-" * 48)
for n in [0.5, 0.9, 0.99, 0.999, 1.0, 1.001, 1.01, 1.1, 2.0]:
    k = len(toggle_fixed_points(1e6, n, N=60000))
    print(f"{n:>8}  {k:>16}   {'LATCH' if k >= 3 else 'no latch'}")

print("\nThe boundary sits exactly at n = 1, and gain does not move it.")


R1  Threshold necessity.  Gain pushed to a = 1e6 throughout.

       n   # steady states   verdict
------------------------------------------------
     0.5                 1   no latch
     0.9                 1   no latch
    0.99                 1   no latch
   0.999                 1   no latch
     1.0                 1   no latch
   1.001                 3   LATCH
    1.01                 3   LATCH
     1.1                 3   LATCH
     2.0                 3   LATCH

The boundary sits exactly at n = 1, and gain does not move it.


The transition is at $n=1$ exactly, and a million-fold gain does not push it below. R1 holds in its
strong form: **cooperativity is not an efficiency, it is a precondition.**

This is the chemical reading of *C0 must be manufactured*. A linear substrate has quantities but no
distinctions. Something has to snap before there is anything to hold.

---
## Step 4 — R2: the gain threshold, and where the falsifiable number comes from

R2 predicted a critical gain. That prediction is worth more if the number is derived rather than
measured, so derive it.

Bistability appears when the symmetric state $u^\*=v^\*$ loses stability. That happens when the
round-trip loop gain reaches 1:

$$\left|f'(u^\*)\right| = 1, \qquad f(x)=\frac{a}{1+x^n}$$

At the symmetric state $u^\* = a/(1+u^{*n})$, so $a = u^\*(1+u^{*n})$. Substituting:

$$\left|f'(u^\*)\right| = \frac{a\,n\,u^{*\,n-1}}{(1+u^{*n})^2} = \frac{n\,u^{*n}}{1+u^{*n}} = 1$$

which rearranges to

$$\boxed{u^{*n} = \frac{1}{n-1}}$$

**This is where R1 comes from as a theorem, not a sweep.** At $n=1$ the right side is undefined and the
condition has no solution — the loop gain can never reach 1, for any $a$. The numerical boundary in
Step 3 was this divergence.

For $n>1$, back-substituting $u^\* = (n-1)^{-1/n}$ gives the onset gain in closed form:

$$\boxed{a_c(n) \;=\; n\,(n-1)^{-\frac{n+1}{n}}}$$

This number could have been otherwise. It is checkable at $0.98\,a_c$ and $1.02\,a_c$ for any $n$.


In [3]:
print("R2  Gain threshold.  Closed form a_c(n) = n (n-1)^(-(n+1)/n)\n")
print(f"{'n':>7}  {'a_c (closed form)':>19}  {'#fp @ 0.98 a_c':>15}  {'#fp @ 1.02 a_c':>15}  match")
print("-" * 82)
hits = 0
rows = [1.02, 1.1, 1.2, 1.5, 2.0, 3.0, 4.0, 8.0, 20.0]
for n in rows:
    ac = a_critical(n)
    lo = len(toggle_fixed_points(0.98 * ac, n, N=60000))
    hi = len(toggle_fixed_points(1.02 * ac, n, N=60000))
    ok = (lo == 1 and hi == 3)
    hits += ok
    print(f"{n:>7}  {ac:>19.6f}  {lo:>15}  {hi:>15}  {'yes' if ok else 'NO'}")
print(f"\n{hits} of {len(rows)} match: one state just below a_c, three just above.")

print("\nAnd the loop gain sitting exactly at the predicted onset:")
for n in [1.2, 1.5, 2.0, 3.0, 8.0]:
    per, rt = loop_gain(a_critical(n), n)
    print(f"   n={n:<5} per-stage |f'| = {per:.8f}   round-trip = {rt:.8f}")
print("\nRound-trip gain = 1.00000000 at onset, for every n. R2 is exact, not approximate.")


R2  Gain threshold.  Closed form a_c(n) = n (n-1)^(-(n+1)/n)

      n    a_c (closed form)   #fp @ 0.98 a_c   #fp @ 1.02 a_c  match
----------------------------------------------------------------------------------
   1.02          2361.712581                1                3  yes
    1.1            89.224391                1                3  yes
    1.2            22.941735                1                3  yes
    1.5             4.762203                1                3  yes
    2.0             2.000000                1                3  yes
    3.0             1.190551                1                3  yes
    4.0             1.013114                1                3  yes
    8.0             0.896096                1                3  yes
   20.0             0.908529                1                3  yes

9 of 9 match: one state just below a_c, three just above.

And the loop gain sitting exactly at the predicted onset:
   n=1.2   per-stage |f'| = 1.00000000   round-trip = 1

$a_c$ matches at every $n$ tested, and the round-trip gain at onset is 1 to eight digits. Note what
$a_c(n)$ does as $n\to1^+$: it diverges like $(n-1)^{-1}$. R1 and R2 are not two requirements, they are
one requirement seen from two sides — **the loop gain must reach unity, and a linear substrate cannot get
there.**

This is the chemical form of Keyes' *gain* criterion, and of C1: a continuation that decays is not a
continuation.

---
## Step 5 — R3: does the loop sign matter, holding everything else fixed?

The sharpest available knockout. Keep the gain, keep the cooperativity, keep the dimension, keep the
degradation. Flip **one arrow** from repression to activation, so the loop sign goes from $+$ to $-$.

If verbs are what matter, the latch should vanish. If "two chemicals with nonlinear feedback" were the
real requirement, it should survive.


In [4]:
a, n = 10.0, 2.0   # both networks get identical parameters

pos = toggle_fixed_points(a, n)
print(f"POSITIVE loop  (repressor + repressor), a={a}, n={n}")
print(f"   steady states: {len(pos)}  ->  {'LATCH' if len(pos) >= 3 else 'no latch'}")
print(f"   the two stable states: u = {pos[0]:.4f} and u = {pos[-1]:.4f}\n")

ends = attractor_count(mixed_rhs, 2, a, n, trials=12)
print(f"NEGATIVE loop  (repressor + activator), a={a}, n={n}  -- identical gain and cooperativity")
print(f"   distinct attractors from 12 random starts: {len(ends)}")
for e in ends:
    print(f"      {e}")
print(f"   ->  {'LATCH' if len(ends) > 1 else 'NO LATCH: one global attractor'}")

print("\nSame gain. Same threshold. Same dimension. Same degradation. One arrow flipped.")
print("Memory is gone.")


POSITIVE loop  (repressor + repressor), a=10.0, n=2.0
   steady states: 3  ->  LATCH
   the two stable states: u = 0.1010 and u = 9.8990

NEGATIVE loop  (repressor + activator), a=10.0, n=2.0  -- identical gain and cooperativity
   distinct attractors from 12 random starts: 1
      (np.float64(0.74), np.float64(3.538))
   ->  NO LATCH: one global attractor

Same gain. Same threshold. Same dimension. Same degradation. One arrow flipped.
Memory is gone.


R3 confirmed, and it is the cleanest of the four because nothing else moved. The latch is not made of
the chemicals and not made of the nonlinearity. It is made of the **loop sign** — which is a property of
the wiring, not of any part.

This is *mutual suppression gives memory*, and it is why nothing is stored "in" a component. It is also
why the electron version and the chemical version are the same verb: in both, the state is the loop's
shape.

---
## Step 6 — R5: what does counting cost?

The Clock verb needs a negative loop with enough phase lag to keep overshooting instead of settling.
Two predictions:

1. **Parity.** Ring of $N$ repressors has loop sign $(-1)^N$. Even $N$ → positive loop → latch. Odd $N$ →
   negative loop → clock.
2. **Dimension.** $N=2$ is the minimum latch, but a 2-species negative loop cannot cycle — with only two
   axes there is no room to lag. So the minimum clock is $N=3$, and **counting costs exactly one axis
   more than memory.**

Both are falsifiable by a single counterexample.


In [5]:
print("R5  Parity: even rings latch, odd rings clock.\n")
for a, n in [(200.0, 2.5), (1000.0, 2.0), (50.0, 3.0)]:
    print(f"  a={a:<7} n={n}")
    for N in [2, 3, 4, 5, 6, 7]:
        amp = oscillation_amplitude(N, a, n)
        sign = "+" if N % 2 == 0 else "-"
        kind = "CLOCK" if amp > 1e-4 else "settles"
        print(f"     N={N}  loop sign {sign}   swing={amp:>9.2e}   {kind}")
    print()


R5  Parity: even rings latch, odd rings clock.

  a=200.0   n=2.5
     N=2  loop sign +   swing= 3.01e-14   settles
     N=3  loop sign -   swing= 1.27e+02   CLOCK
     N=4  loop sign +   swing= 3.07e-14   settles
     N=5  loop sign -   swing= 1.98e+02   CLOCK
     N=6  loop sign +   swing= 3.24e-12   settles
     N=7  loop sign -   swing= 2.00e+02   CLOCK

  a=1000.0  n=2.0
     N=2  loop sign +   swing= 4.13e-11   settles
     N=3  loop sign -   swing= 1.21e-09   settles
     N=4  loop sign +   swing= 5.38e-11   settles
     N=5  loop sign -   swing= 9.73e+02   CLOCK
     N=6  loop sign +   swing= 5.29e-11   settles
     N=7  loop sign -   swing= 9.99e+02   CLOCK

  a=50.0    n=3.0
     N=2  loop sign +   swing= 3.03e-14   settles
     N=3  loop sign -   swing= 3.75e+01   CLOCK
     N=4  loop sign +   swing= 4.26e-14   settles
     N=5  loop sign -   swing= 4.94e+01   CLOCK
     N=6  loop sign +   swing= 1.19e-12   settles
     N=7  loop sign -   swing= 5.00e+01   CLOCK



In [6]:
print("And the even rings: do they latch (two attractors) or merely settle to one state?\n")
a, n = 200.0, 2.5
for N in [2, 4, 6]:
    ends = attractor_count(ring_rhs, N, a, n, trials=12)
    print(f"  N={N}: distinct attractors from 12 random starts = {len(ends)}  "
          f"-> {'LATCH' if len(ends) > 1 else 'single state'}")
    for e in sorted(ends):
        print(f"       {e}")
print("\nEven rings hold an alternating high/low pattern, two ways round. That is a latch.")
print("Minimum latch: N=2.   Minimum clock: N=3.   Counting costs one axis more than memory.")


And the even rings: do they latch (two attractors) or merely settle to one state?

  N=2: distinct attractors from 12 random starts = 2  -> LATCH
       (np.float64(0.0), np.float64(200.0))
       (np.float64(200.0), np.float64(0.0))
  N=4: distinct attractors from 12 random starts = 2  -> LATCH
       (np.float64(0.0), np.float64(200.0), np.float64(0.0), np.float64(200.0))
       (np.float64(200.0), np.float64(0.0), np.float64(200.0), np.float64(0.0))
  N=6: distinct attractors from 12 random starts = 2  -> LATCH
       (np.float64(0.0), np.float64(200.0), np.float64(0.0), np.float64(200.0), np.float64(0.0), np.float64(200.0))
       (np.float64(200.0), np.float64(0.0), np.float64(200.0), np.float64(0.0), np.float64(200.0), np.float64(0.0))

Even rings hold an alternating high/low pattern, two ways round. That is a latch.
Minimum latch: N=2.   Minimum clock: N=3.   Counting costs one axis more than memory.


---
## Step 6b — The anomaly, and the sharper law it forced

The parity sweep has **one miss**. At $a=1000,\;n=2.0$ the 3-ring did not oscillate, while the 5-ring and
7-ring did. Parity alone predicted a clock there.

A miss is the useful kind of result if it has an exact cause, so run it down. Linearize the ring at its
symmetric state. The Jacobian is circulant, so its eigenvalues are

$$\lambda_k = -1 - G\,e^{2\pi i k/N},\qquad G=\left|f'(u^*)\right|>0,\quad k=0,\dots,N-1$$

The state goes unstable when some $\operatorname{Re}\lambda_k>0$, i.e. $G\cos(2\pi k/N)<-1$. For odd $N$ the
most negative available cosine is $-\cos(\pi/N)$, so the ring oscillates precisely when

$$\boxed{G \;>\; \sec\!\left(\frac{\pi}{N}\right)}$$

and since $G\to n$ as $a\to\infty$, the requirement on cooperativity is $n>\sec(\pi/N)$:

| $N$ | $\sec(\pi/N)$ | minimum cooperativity to clock |
|---|---|---|
| 3 | **2.000** | $n>2$ |
| 5 | 1.236 | $n>1.236$ |
| 7 | 1.110 | $n>1.110$ |
| 11 | 1.042 | $n>1.042$ |

**That is the miss, exactly.** At $n=2.0$ the 3-ring sits *on* its threshold, not above it. The 5- and
7-rings sit well above theirs. Parity was necessary but not sufficient, and the sufficiency condition is
a closed form.


In [7]:
print("Testing n_crit(N) = sec(pi/N).  Gain a = 1e7 so that G -> n.\n")
print(f"{'N':>3} {'sec(pi/N)':>11} | {'just below':>34} | {'just above':>34}")
print("-" * 90)
hits = 0
for N in [3, 5, 7, 9, 11]:
    nc = 1.0 / np.cos(np.pi / N)
    a = 1e7
    lo, hi = nc * 0.97, nc * 1.06
    alo = oscillation_amplitude(N, a, lo, T=6000)
    ahi = oscillation_amplitude(N, a, hi, T=6000)
    ok = (alo < 1e-4 and ahi > 1e-4)
    hits += ok
    print(f"{N:>3} {nc:>11.6f} | n={lo:.4f} G={loop_gain(a,lo)[0]:.3f} swing={alo:8.1e} "
          f"| n={hi:.4f} G={loop_gain(a,hi)[0]:.3f} swing={ahi:8.1e}  {'ok' if ok else 'NO'}")
print(f"\n{hits} of 5 match: flat below sec(pi/N), oscillating above.")

print("\nAnd the anomaly itself -- N=3 at a=1000, walked across G = 2:")
for n in [1.95, 2.00, 2.05, 2.50]:
    G = loop_gain(1000.0, n)[0]
    sw = oscillation_amplitude(3, 1000.0, n, T=6000)
    print(f"   n={n:<5} G={G:.4f} {'<' if G < 2 else '>'} 2   swing={sw:.2e}   {'CLOCK' if sw > 1e-4 else 'flat'}")
print("\nThe 3-ring crosses into oscillation exactly at G = 2 = sec(pi/3). The miss was the law, not noise.")


Testing n_crit(N) = sec(pi/N).  Gain a = 1e7 so that G -> n.

  N   sec(pi/N) |                         just below |                         just above
------------------------------------------------------------------------------------------
  3    2.000000 | n=1.9400 G=1.940 swing= 4.9e-10 | n=2.1200 G=2.120 swing= 5.7e+06  ok
  5    1.236068 | n=1.1990 G=1.199 swing= 3.1e-09 | n=1.3102 G=1.310 swing= 2.0e+04  ok
  7    1.109916 | n=1.0766 G=1.076 swing= 4.8e-09 | n=1.1765 G=1.176 swing= 1.7e+05  ok
  9    1.064178 | n=1.0323 G=1.032 swing= 5.7e-09 | n=1.1280 G=1.128 swing= 1.2e+06  ok
 11    1.042217 | n=1.0110 G=1.011 swing= 6.2e-09 | n=1.1048 G=1.105 swing= 3.5e+06  ok

5 of 5 match: flat below sec(pi/N), oscillating above.

And the anomaly itself -- N=3 at a=1000, walked across G = 2:
   n=1.95  G=1.9298 < 2   swing=2.13e-11   flat
   n=2.0   G=1.9801 < 2   swing=1.21e-09   flat
   n=2.05  G=2.0303 > 2   swing=6.25e+01   CLOCK
   n=2.5   G=2.4820 > 2   swing=7.52e+02   CLOCK

The

$\sec(\pi/N)$ holds at 5 of 5, and the anomaly resolves on the number.

Now look at what the formula says, because this is the part that matters for the framework:

$$\sec(\pi/N)\;\longrightarrow\;1 \quad\text{as}\quad N\to\infty$$

**The longer the loop, the less nonlinearity each stage needs.** A 3-ring demands a hard threshold at every
stage ($n>2$). An 11-ring clocks with almost none ($n>1.042$), and in the limit the requirement collapses to
the *same* unity-gain condition the latch needed. The threshold requirement does not disappear — it gets
distributed around the loop.

So the irreducible verb requirement, across both capabilities, is one thing: **round-trip gain must exceed
1.** Everything else — how much cooperativity, how many species — is how that single requirement gets paid
for at a given loop length. Depth buys relaxed local requirements, which is the same statement as more
arguments exposed on the same verb.


---
## Step 7 — The transfer check: do both laws hold in silicon?

Here is the part that decides whether the verb map is a real map or a set of resemblances.

Both laws were derived from the requirement table and confirmed in a chemical reaction network. If the verb
set is substrate-independent, the same two laws should govern **inverter rings** — a substrate with nothing
chemically in common.

They do, and both are standard engineering knowledge:

**Law 1 — parity.**

| $N$ inverting stages | Loop sign | Silicon | Chemistry |
|---|---|---|---|
| 2 | $+$ | Cross-coupled inverter pair → **SRAM cell / latch** | Two mutual repressors → **toggle** |
| 3 | $-$ | Three-stage **ring oscillator** | Three-repressor oscillator |
| 4 | $+$ | Latches — which is why ring oscillators are never built even-staged | Latches, alternating |
| 5 | $-$ | Ring oscillator | Oscillator |

**Law 2 — the gain condition.** The requirement that an $N$-stage ring oscillator have per-stage gain
exceeding $\sec(\pi/N)$ is the textbook design rule for CMOS ring oscillators, derived there from the
Barkhausen phase criterion. It is the *same* circulant eigenvalue calculation, in the same form, with
transconductance in place of Hill cooperativity. The three-stage minimum needing gain $>2$ is the case every
circuit designer knows.

The biology side is the fair test, because those objects were engineered independently and before any of
this. The smallest synthetic genetic memory ever built is a **two**-repressor toggle (Gardner, Cantor &
Collins 2000). The smallest synthetic genetic clock is a **three**-repressor ring, the repressilator
(Elowitz & Leibler 2000) — and its designers had to engineer *strong cooperative* repression to make it run,
which is $n>2$ at $N=3$. Both sit exactly at the predicted minimum, for the predicted reason.

Three substrates. One parity law. One gain law. One dimension cost.


---
## Step 8 — Scoring, honestly

The discipline demanded: a right object arrives with a number that could have been otherwise and matches.
A resemblance arrives with a number chosen to fit a target. Score:

| Prediction | From verbs alone | Result | Number |
|---|---|---|---|
| R1 Threshold necessary | cooperativity required | **confirmed, strong form** | boundary at $n=1$ exactly; $10^6$ gain does not move it |
| R2 Gain threshold exists | a critical production rate | **confirmed, derived** | $a_c=n(n-1)^{-(n+1)/n}$, 9 of 9; round-trip gain $=1.00000000$ at onset |
| R3 Loop sign decides | $+$ latches, $-$ does not | **confirmed, clean knockout** | 3 states → 1, everything else held fixed |
| R5a Parity | even latch, odd clock | **partly confirmed** | latch side 9 of 9 exact; clock side 17 of 18 raw |
| R5a′ the one miss | — | **resolved by a sharper law** | $G>\sec(\pi/N)$, 5 of 5; explains the miss at $G=1.98<2$ |
| R5b Dimension cost | clock needs one axis more | **confirmed** | min latch $N=2$, min clock $N=3$ |

The miss is the most informative line in the table. Parity turned out to be necessary and not sufficient,
and chasing the failure produced $\sec(\pi/N)$ — a second closed form that also transfers to silicon. A
resemblance would not have failed in a way that yielded a formula.

**What is not new here.** None of the individual facts are. That the mutual-repression toggle needs $n>1$ is
known in systems biology. Loop-gain-equals-one is the standard saddle-node condition. $\sec(\pi/N)$ is the
textbook ring-oscillator design rule in circuits and appears in the gene-ring literature too. Every cell
above would be recognized by a specialist *in that substrate*.

**What is new is the direction the derivation ran.** The requirement list came from electrons. It was
written down before the chemical model was, and with no chemistry it predicted that the network would need
cooperativity, would have a critical gain, would die on a sign flip, would need one more species to count
than to remember, and — once the miss was chased — would need a cooperativity that *falls* with loop
length. Six predictions, six confirmations, three exact numbers, one of them found by failure.

That is the difference between a verb map and a metaphor. A metaphor gets the shapes right and carries no
number. This carried two closed forms and a parity, and all three transferred across substrates.

**Where it could still fail.** The clean results assumed symmetric parameters and equal degradation rates.
Asymmetric rings, explicit delay, and substrates with conserved total mass — where production is not free —
are untested, and are where to look for the boundary. The no-cloning limit on the **Copy** verb is the one
place the verb set is already known to change with depth; whether any other verb has such a boundary is
open.


---
## Step 9 — What this settles in the constraint chain

The framing question was: *a machine that retains paths must have curvature, a machine that retains state
must have a cycle.* This puts numbers on the second half.

- **A machine that retains state must have a cycle** — confirmed, and the cycle is a *positive* loop,
  minimum length 2. The state is the loop's shape, not any part's content.
- **A machine that counts must have a cycle it never closes** — a negative loop, minimum length 3, with
  per-stage gain above $\sec(\pi/N)$. The extra axis is the price of the phase lag.
- **One requirement underneath both** — round-trip gain $>1$. At the latch it is paid in one step; at the
  clock it is paid around the loop, and a longer loop pays less per stage. $\sec(\pi/N)\to1$ says the two
  capabilities share a single floor and differ only in how they reach it.
- **Both must pay.** Every state above is actively held against degradation. Switching the latch dissipates;
  the hysteresis area is the work. That is Landauer's $kT\ln2$ in chemical clothing, and it is the same line
  as *transformation must pay a price to prove something changed.*

The verb map is not a picture of parts. It is a list of what any arrangement must contain to make a given
continuation lawful. It predicts across substrates because it was never about the substrate.

$$\boxed{\;\text{memory}=\text{positive loop},\;N\!\ge\!2\qquad\text{counting}=\text{negative loop},\;N\!\ge\!3,\;G>\sec(\pi/N)\;}$$
